# Hands-On Lab: Auto Loader with Payments & Orders

###
landing (raw files) 
   ↓ Auto Loader

bronze (raw delta tables)
   ↓ cleaning/dedup

silver (clean tables)
   ↓ aggregation
   
gold (business metrics)



## 🏗️ STEP 0 — Lab Setup (Paths & Structure)

In [0]:
landing_path_payments = "abfss://gizmobox@optronstore.dfs.core.windows.net/landing/operational_data/payments/"
landing_path_orders   = "abfss://gizmobox@optronstore.dfs.core.windows.net/landing/operational_data/orders/"

checkpoint_base = "/mnt/checkpoints/gizmobox/"

In [0]:
storage = "optronstore"
container = "gizmobox"

base_path = f"abfss://{container}@{storage}.dfs.core.windows.net/operational_data/"

orders_path = base_path + "orders/"
payments_path = base_path + "payments/"
addresses_path = base_path + "addresses/"

checkpoint_base = f"abfss://{container}@{storage}.dfs.core.windows.net/checkpoints/"

📥 STEP 1 — Simulate Incoming Data (Landing Layer)

👉 Run this ONCE to create sample files

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("payment_id", IntegerType(), True),
    StructField("order_id", IntegerType(), True),
    StructField("payment_timestamp", TimestampType(), True),
    StructField("payment_status", IntegerType(), True),
    StructField("payment_method", StringType(), True)
])

payments_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema)  # 👈 key fix
    .option("cloudFiles.schemaLocation",
        "abfss://gizmobox@optronstore.dfs.core.windows.net/checkpoints/payments_delta/schema")
    .load("abfss://gizmobox@optronstore.dfs.core.windows.net/operational_data/payments/")
)

## ⚡ STEP 2 — Auto Loader → Bronze (Streaming Ingestion)
   🔹 2.1 Payments Pipeline

In [0]:
from pyspark.sql import Row
import json
payments_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation",
        "abfss://gizmobox@optronstore.dfs.core.windows.net/checkpoints/payments_delta/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load("abfss://gizmobox@optronstore.dfs.core.windows.net/landing/operational_data/payments/")
)

(payments_stream.writeStream
    .format("delta")
    .option("checkpointLocation",
        "abfss://gizmobox@optronstore.dfs.core.windows.net/checkpoints/payments_delta/checkpoint")
    .trigger(availableNow=True)
    .toTable("gizmobox.bronze.payments_delta")
)

### 🔹 2.2 Orders Pipeline

In [0]:
from pyspark.sql.functions import col, to_date

orders_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation",
        "abfss://gizmobox@optronstore.dfs.core.windows.net/checkpoints/orders_delta/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load("abfss://gizmobox@optronstore.dfs.core.windows.net/operational_data/orders/")
)

orders_stream = (orders_stream
    .withColumn("order_id", col("order_id").cast("int"))
    .withColumn("order_date", to_date("order_date"))
)

(orders_stream.writeStream
    .format("delta")
    .option("checkpointLocation",
        "abfss://gizmobox@optronstore.dfs.core.windows.net/checkpoints/orders_delta/checkpoint")
    .trigger(availableNow=True)
    .toTable("gizmobox.bronze.orders_delta")
)

## ✅ STEP 3 — Validate Bronze Tables


In [0]:
%sql
SELECT COUNT(*) FROM gizmobox.bronze.payments_delta;
SELECT COUNT(*) FROM gizmobox.bronze.orders_delta;

In [0]:
.trigger(availableNow=True)

In [0]:
%sql
select * from _sqldf

In [0]:
display(_sqldf)

## 🔄 ⚡ STEP 4 — Auto Loader (Addresses TSV → Delta)
🔹 Clean + Deduplicate
Payments


In [0]:
addresses_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("delimiter", "\t")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.schemaLocation", checkpoint_base + "addresses_delta/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(addresses_path)
)

(addresses_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_base + "addresses_delta/checkpoint")
    .trigger(availableNow=True)
    .toTable("gizmobox.bronze.addresses_delta")
)

In [0]:
%sql

SELECT * FROM gizmobox.bronze.addresses_delta;

In [0]:
%sql
SELECT * FROM gizmobox.silver.orders;

### ✅ STEP 5 — Validate Bronze Delta Tables

In [0]:
%sql
SELECT 
    (SELECT COUNT(*) FROM gizmobox.bronze.payments_delta) AS payment_count,
    (SELECT COUNT(*) FROM gizmobox.bronze.orders_delta) AS order_count,
    (SELECT COUNT(*) FROM gizmobox.bronze.addresses_delta) AS address_count;

### 🔁 STEP 6 — Test Incremental Load

👉 Upload new file(s):  Just upload new orders as json files under ADLS Folder : gizmobox:operational_data\orders  and then rerun step 3 autoloader on 

In [0]:
display(_sqldf)

In [0]:
%sql
--Check the history of the bronze table for orders
DESCRIBE HISTORY gizmobox.bronze.orders_delta;

## 🔄 STEP 7 — Silver Layer (From Delta Only)
Payments

In [0]:
from pyspark.sql.functions import col, lit, to_date

payments = spark.readStream.table("gizmobox.bronze.payments_delta")

payments_transformed = (payments
    # Cast IDs
    .withColumn("payment_id", col("payment_id").cast("int"))
    .withColumn("order_id", col("order_id").cast("int"))
    
    # Map existing fields
    .withColumn("payment_status", col("status").cast("string"))
    
    # Create missing columns (since they don’t exist)
    .withColumn("payment_date", lit(None).cast("date"))
    .withColumn("payment_time", lit(None).cast("string"))
    .withColumn("payment_method", lit("unknown"))
    
    # Select final schema
    .select(
        "payment_id",
        "order_id",
        "payment_date",
        "payment_time",
        "payment_status",
        "payment_method"
    )
)

(payments_transformed.writeStream
    .format("delta")
    .option("checkpointLocation",
        "abfss://gizmobox@optronstore.dfs.core.windows.net/checkpoints/payments_silver_v2")
    .toTable("gizmobox.silver.payments_v2")
)

In [0]:
%sql
SELECT * FROM gizmobox.silver.payments_v2 LIMIT 10;

Validate
👉 If data is there → your schema is wrong upstream

In [0]:
spark.read.table("gizmobox.bronze.payments_delta") \
    .select("_rescued_data") \
    .display()

Orders

In [0]:
from pyspark.sql.functions import col, from_json, explode
from pyspark.sql.types import *

orders = spark.readStream.table("gizmobox.bronze.orders_delta")

# Define schema of items JSON
items_schema = ArrayType(StructType([
    StructField("item_id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("price", LongType(), True),
    StructField("quantity", LongType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("color", StringType(), True)
]))

orders_parsed = (orders
    # Parse JSON string → array<struct>
    .withColumn("items_array", from_json(col("items"), items_schema))
    
    # Explode array → multiple rows
    .withColumn("item", explode("items_array"))
)

orders_transformed = (orders_parsed
    # Cast base fields
    .withColumn("order_id", col("order_id").cast("bigint"))
    .withColumn("customer_id", col("customer_id").cast("bigint"))

    # Extract nested fields
    .withColumn("item_id", col("item.item_id"))
    .withColumn("name", col("item.name"))
    .withColumn("price", col("item.price"))
    .withColumn("quantity", col("item.quantity"))
    .withColumn("category", col("item.category"))
    .withColumn("brand", col("item.brand"))
    .withColumn("color", col("item.color"))

    # Select final schema
    .select(
        "order_id",
        "order_date",
        "order_status",
        "customer_id",
        "item_id",
        "name",
        "price",
        "quantity",
        "category",
        "brand",
        "color"
    )
)

(orders_transformed.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "abfss://gizmobox@optronstore.dfs.core.windows.net/checkpoints/orders_silver"
    )
    .toTable("gizmobox.silver.orders")
)

In [0]:
spark.read.table("gizmobox.bronze.orders_delta").printSchema()

In [0]:
%sql
SELECT 
    (SELECT COUNT(*) FROM gizmobox.silver.payments) AS payment_count,
    (SELECT COUNT(*) FROM gizmobox.silver.orders) AS order_count,
    (SELECT COUNT(*) FROM gizmobox.silver.addresses) AS address_count;

In [0]:
spark.read.table("gizmobox.bronze.addresses_delta").printSchema()

In [0]:
spark.read.table("gizmobox.silver.addresses").printSchema()

In [0]:
from pyspark.sql.functions import col

addresses = spark.readStream.table("gizmobox.bronze.addresses_delta")

addresses_transformed = (addresses
    # Cast customer_id to bigint (common join key)
    .withColumn("customer_id", col("customer_id").cast("bigint"))

    # Ensure all string fields (safe even if already string)
    .withColumn("address_type", col("address_type").cast("string"))
    .withColumn("address_line_1", col("address_line_1").cast("string"))
    .withColumn("city", col("city").cast("string"))
    .withColumn("state", col("state").cast("string"))
    .withColumn("postcode", col("postcode").cast("string"))

    # Select only required columns (drop _rescued_data)
    .select(
        "customer_id",
        "address_type",
        "address_line_1",
        "city",
        "state",
        "postcode"
    )
)

(addresses_transformed.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "abfss://gizmobox@optronstore.dfs.core.windows.net/checkpoints/addresses_silver"
    )
    .toTable("gizmobox.silver.addresses")
)

In [0]:
from pyspark.sql.functions import col, when, max as spark_max

addresses = spark.readStream.table("gizmobox.bronze.addresses_delta")

addresses_casted = (addresses
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("postcode", col("postcode").cast("int"))
)

addresses_pivoted = (addresses_casted
    .groupBy("customer_id")
    .agg(
        # SHIPPING
        spark_max(when(col("address_type") == "shipping", col("address_line_1"))).alias("shipping_address_1"),
        spark_max(when(col("address_type") == "shipping", col("city"))).alias("shipping_city"),
        spark_max(when(col("address_type") == "shipping", col("state"))).alias("shipping_state"),
        spark_max(when(col("address_type") == "shipping", col("postcode"))).alias("shipping_postcode"),

        # BILLING
        spark_max(when(col("address_type") == "billing", col("address_line_1"))).alias("billing_address_1"),
        spark_max(when(col("address_type") == "billing", col("city"))).alias("billing_city"),
        spark_max(when(col("address_type") == "billing", col("state"))).alias("billing_state"),
        spark_max(when(col("address_type") == "billing", col("postcode"))).alias("billing_postcode")
    )
)

(addresses_pivoted.writeStream
    .format("delta")
    .outputMode("complete")
    .option(
        "checkpointLocation",
        "abfss://gizmobox@optronstore.dfs.core.windows.net/checkpoints/addresses_silver"
    )
    .toTable("gizmobox.silver.addresses")
)

⚡ STEP 1 — Read Silver Tables (Batch for simplicity)

In [0]:
import pyspark.sql.functions as F

orders = spark.read.table("gizmobox.silver.orders")
payments = spark.read.table("gizmobox.silver.payments_v2")
addresses = spark.read.table("gizmobox.silver.addresses")

🔗 STEP 2 — Join Everything

In [0]:
customer_360 = (orders
    .join(payments, "order_id", "left")
    .join(addresses, "customer_id", "left")
)

STEP 3 — Create Business Metrics

In [0]:
customer_metrics = (customer_360
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("price").alias("total_revenue"),
        F.sum("quantity").alias("total_items"),
        F.first("shipping_city").alias("city"),
        F.first("shipping_state").alias("state")
    )
)

💾 STEP 4 — Write GOLD Table

In [0]:
(customer_metrics.write
    .mode("overwrite")
    .saveAsTable("gizmobox.gold.customer_360")
)

In [0]:
%sql
SELECT * FROM gizmobox.gold.customer_360 LIMIT 10;

Optional GOLD Tables (Highly Valuable)
🟢 Orders by State

In [0]:
orders_by_state = (customer_360
    .groupBy("shipping_state")
    .agg(F.countDistinct("order_id").alias("orders"))
)

orders_by_state.write.mode("overwrite").saveAsTable("gizmobox.gold.orders_by_state")

In [0]:
revenue_by_payment = (customer_360
    .groupBy("payment_method")
    .agg(F.sum("price").alias("revenue"))
)

revenue_by_payment.write.mode("overwrite").saveAsTable("gizmobox.gold.revenue_by_payment")